# 02 · Бейзлайн: TF-IDF + мета-ознаки → регресія → оптимізація порогів

**Ідея:** QWK — порядкова метрика, тому передбачаємо неперервне значення
і підбираємо пороги (`OptimizedRounder`) на OOF-передбаченнях. Це стабільно
краще за пряму класифікацію.

Ознаки: TF-IDF (слова + символи) → усічення SVD, статистики тексту, кількість фото.
Модель: LightGBM (регресія).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import *
from src.utils import seed_everything, qwk, OptimizedRounder, write_submission
from src.data import load_data, build_image_index, add_photo_count, add_folds, text_meta_features

seed_everything(SEED)
pd.set_option("display.max_colwidth", 200)
sns.set_theme(style="whitegrid")

In [ ]:
import lightgbm as lgb
from scipy.sparse import hstack, csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

train = pd.read_parquet(PROCESSED_DIR / "train_meta.parquet")
test = pd.read_parquet(PROCESSED_DIR / "test_meta.parquet")
print(train.shape, test.shape)

## 1. Текстові ознаки: TF-IDF → SVD

In [ ]:
N_SVD = 120
all_text = pd.concat([train[TEXT_COL], test[TEXT_COL]])

tfidf_word = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=50_000, sublinear_tf=True)
tfidf_char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=5, max_features=80_000, sublinear_tf=True)

X_text = hstack([tfidf_word.fit_transform(all_text), tfidf_char.fit_transform(all_text)]).tocsr()
svd = TruncatedSVD(n_components=N_SVD, random_state=SEED)
X_svd = svd.fit_transform(X_text)
print("explained var:", svd.explained_variance_ratio_.sum().round(3))

svd_cols = [f"svd_{i}" for i in range(N_SVD)]
train_svd = pd.DataFrame(X_svd[: len(train)], columns=svd_cols)
test_svd = pd.DataFrame(X_svd[len(train):], columns=svd_cols)

## 2. Матриця ознак

In [ ]:
meta_cols = ["n_photos", "desc_len", "desc_words", "desc_upper_ratio", "desc_excl",
             "desc_digits", "desc_is_empty", "desc_non_ascii"]

X_train = pd.concat([train[meta_cols].reset_index(drop=True), train_svd], axis=1)
X_test = pd.concat([test[meta_cols].reset_index(drop=True), test_svd], axis=1)
y = train[TARGET_COL].values
X_train.shape, X_test.shape

## 3. Крос-валідація LightGBM (регресія)

In [ ]:
params = dict(
    objective="regression", learning_rate=0.02, num_leaves=31,
    feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=1,
    lambda_l2=1.0, min_data_in_leaf=20, verbose=-1, seed=SEED,
)

oof = np.zeros(len(train))
pred_test = np.zeros(len(test))

for f in range(N_FOLDS):
    tr_idx, va_idx = np.where(train["fold"] != f)[0], np.where(train["fold"] == f)[0]
    dtr = lgb.Dataset(X_train.iloc[tr_idx], y[tr_idx])
    dva = lgb.Dataset(X_train.iloc[va_idx], y[va_idx])
    model = lgb.train(params, dtr, num_boost_round=5000, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(200, verbose=False)])
    oof[va_idx] = model.predict(X_train.iloc[va_idx])
    pred_test += model.predict(X_test) / N_FOLDS
    print(f"fold {f}: best_iter={model.best_iteration}  rmse={np.sqrt(np.mean((oof[va_idx]-y[va_idx])**2)):.4f}")

## 4. Оптимізація порогів та QWK

In [ ]:
labels = tuple(sorted(train[TARGET_COL].unique()))  # (1, 2, 3, 4)
rounder = OptimizedRounder(labels=labels).fit(oof, y)
print("thresholds:", rounder.coef_.round(3))
print("OOF QWK (optimised):", round(qwk(y, rounder.predict(oof)), 4))
print("OOF QWK (naive round):", round(qwk(y, np.clip(np.round(oof), y.min(), y.max())), 4))

In [ ]:
# Save OOF / test predictions for stacking later
np.save(PROCESSED_DIR / "oof_baseline.npy", oof)
np.save(PROCESSED_DIR / "test_baseline.npy", pred_test)

## 5. Сабмішн

In [ ]:
write_submission(test[ID_COL], rounder.predict(pred_test), "01_baseline_tfidf_lgb");

## Висновки

_(OOF QWK, LB score, що працює / не працює, наступні кроки)_